##Libraries

In [1]:
%%capture
!pip install transformers torch sentence-transformers bert-score
!pip install openai
!pip install requests

##1. BERTScore

###Bidirectional Encoder Representations from Transformers

Compares generated text with a gold/reference answer.

In [2]:
from bert_score import score

candidate = """
Paris is the capital of France and has a population of 50 million.
"""

reference = """
Paris is the capital city of France. Its population is approximately 2.1 million.
"""

P, R, F1 = score(
    [candidate],
    [reference],
    model_type="distilbert-base-uncased",
    lang="en"
)

print(f"BERTScore F1: {F1.mean().item():.4f}")

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


BERTScore F1: 0.9097


Interpretation

| Score     | Meaning                 |
| --------- | ----------------------- |
| >0.90     | Very similar            |
| 0.80–0.90 | Mostly correct          |
| <0.80     | Potential hallucination |


##2. SelfCheckGPT

Generate multiple answers and measure agreement.

In [3]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')

responses = [
    "Paris is the capital of France.",
    "The capital of France is Paris.",
    "France's capital city is Paris.",
    "Paris serves as France's capital."
]

embeddings = model.encode(responses)

sims = []

for i in range(len(embeddings)):
    for j in range(i+1, len(embeddings)):
        sims.append(
            cosine_similarity(
                [embeddings[i]],
                [embeddings[j]]
            )[0][0]
        )

consistency_score = np.mean(sims)

print(f"Consistency Score: {consistency_score:.4f}")

Consistency Score: 0.9617


Interpretation

0.90+  => very consistent

0.75-0.90 => moderate confidence

<0.75 => likely hallucination

##3. FactScore-Style Verification

Extract claims and verify against retrieved evidence.

In [4]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer('all-MiniLM-L6-v2')

claim = "Paris has a population of 50 million."

evidence = """
Paris is the capital of France.
Its population is approximately 2.1 million.
"""

claim_emb = model.encode(claim, convert_to_tensor=True)
evidence_emb = model.encode(evidence, convert_to_tensor=True)

score = util.cos_sim(claim_emb, evidence_emb)

print("FactScore-like similarity:",
      score.item())

FactScore-like similarity: 0.8239729404449463


Typical Interpretation

| FactScore   | Interpretation                         |
| ----------- | -------------------------------------- |
| 0.90 – 1.00 | Excellent factual grounding            |
| 0.75 – 0.90 | Mostly factual, few unsupported claims |
| 0.50 – 0.75 | Significant factual issues             |
| < 0.50      | Many hallucinations                    |
| 0.00        | Completely unsupported                 |


##4. HHEM (Vectara)

Vectara publishes hallucination-evaluation models on HuggingFace.

In [12]:
%%capture
!pip install sentencepiece tiktoken

###Resolving Error Due to Version Issue

In [14]:
import transformers
import tokenizers
import sentencepiece

print(transformers.__version__)
print(tokenizers.__version__)

5.10.2
0.22.2


In [15]:
import transformers
print(transformers.__file__)
print(transformers.__version__)

/usr/local/lib/python3.12/dist-packages/transformers/__init__.py
5.10.2


In [ ]:
!pip uninstall -y transformers tokenizers

!pip install transformers==4.46.3
!pip install tokenizers==0.20.3
!pip install sentencepiece

Found existing installation: transformers 5.10.2
Uninstalling transformers-5.10.2:
  Successfully uninstalled transformers-5.10.2
Found existing installation: tokenizers 0.22.2
Uninstalling tokenizers-0.22.2:
  Successfully uninstalled tokenizers-0.22.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 53.4 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.18.0
    Uninstalling huggingface_hub-1.18.0:
      Successfully uninstalled huggingface_hub-1.18.0


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 377, in run
    requirement_set = resolver.resolve(
                      ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/resolution/resolvelib/resolver.py", line 95, in resolve
    result = self._result = resolver.resolve(
                            ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_vendor/resolvelib/resolvers.py", line 546, in resolve
    state = resolution.resolve(requirements, max_rounds=max_rounds)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

###Actual Implementation

In [6]:
from transformers import AutoConfig

cfg = AutoConfig.from_pretrained(
    "vectara/hallucination_evaluation_model",
    trust_remote_code=True
)

print(cfg)

You are using a model of type HHEMv2Config to instantiate a model of type HHEMv2. This is not supported for all configurations of models and can yield errors.


HHEMv2Config {
  "_name_or_path": "vectara/hallucination_evaluation_model",
  "architectures": [
    "HHEMv2ForSequenceClassification"
  ],
  "auto_map": {
    "AutoConfig": "vectara/hallucination_evaluation_model--configuration_hhem_v2.HHEMv2Config",
    "AutoModelForSequenceClassification": "vectara/hallucination_evaluation_model--modeling_hhem_v2.HHEMv2ForSequenceClassification"
  },
  "id2label": {
    "0": "hallucinated",
    "1": "consistent"
  },
  "label2id": null,
  "model_type": "HHEMv2",
  "torch_dtype": "float32",
  "transformers_version": "4.46.3"
}



In [7]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "vectara/hallucination_evaluation_model",
    trust_remote_code=True
)

print(type(model))

You are using a model of type HHEMv2Config to instantiate a model of type HHEMv2. This is not supported for all configurations of models and can yield errors.


modeling_hhem_v2.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/vectara/hallucination_evaluation_model:
- modeling_hhem_v2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/439M [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

<class 'transformers_modules.vectara.hallucination_evaluation_model.8e4a2e6e96c708cc76c2344f7e4757df2515292c.modeling_hhem_v2.HHEMv2ForSequenceClassification'>


In [8]:
print(dir(model))

['T_destination', '__annotations__', '__call__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattr__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setstate__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_apply', '_assisted_decoding', '_auto_class', '_autoset_attn_implementation', '_backward_compatibility_gradient_checkpointing', '_backward_hooks', '_backward_pre_hooks', '_beam_search', '_buffers', '_call_impl', '_check_and_enable_flash_attn_2', '_check_and_enable_sdpa', '_compiled_call_impl', '_constrained_beam_search', '_contrastive_search', '_convert_head_mask_to_5d', '_copy_lm_head_original_to_resized', '_create_repo', '_dispatch_accelerate_model', '_dola_decoding', '_expand_inputs_for_generation', '_extract_past_from_model_output', '_forward_hooks'

In [9]:
import inspect

print(inspect.signature(model.predict))

(text_pairs)


In [10]:
result = model.predict([
    (
        "Paris is the capital of France.",
        "Paris is the capital of France."
    )
])

print(result)

tensor([0.8541])


In [11]:
result = model.predict([
    (
        "Paris is the capital of France.",
        "Tokyo is the capital of France."
    )
])

print(result)

tensor([0.0125])
